# Tutorial 10 - Simple ReAct Agent from Scratch 

**Course:** SYSC 4415 - Introduction to Machine Learning

**Semester:** Winter 2026

**Created by:** [Igor Bogdanov](mailto:igorbogdanov@cmailcarleton.ca), updated March 2026

**Originally Inspired by:** [NVIDIA GTC 2025](https://www.nvidia.com/gtc/) Agentic Session and https://react-lm.github.io/

---

This tutorial builds a ReAct agent from scratch covering four areas: agent core components and state management, the Thought→Action→Observation→Answer reasoning cycle, tool orchestration via callable functions, and an applied case study using a measurement conversion system.


This notebook demonstrates key concepts in building a basic ReAct agent:
---
1. **AI Agent Core Components**
   - Essential elements that compose an intelligent agent
   - Prompt engineering and message processing techniques
   - Conversation state and dialogue management
---
2. **Reasoning-Action Framework**
   - Cognitive process cycle: Thought → Action → Observation → Answer cycle
   - Structured reasoning process
   - Tool utilization and output interpretation
---
3. **Tool Orchestration**
   - Creating callable functions (model_memory, apply_conversion)
   - Processing external data and maintaining system state
---
4. **Applied Agent Development**
   - Case study: Measurement Conversion System
   - Multi-step problem decomposition
   - Automated reasoning with integrated capabilities
---

<img src="assets/react_agent_flowchart.png"  width="20%;" style="background-color:Gainsboro; padding:5%;"/>

## Imports and Initialization

Install and configure the llm-connector package from pip and scaffold the project workspace. All standard library dependencies are imported here.

### GitHub Repo: https://github.com/isbogdanov/llm_connector


In [ ]:
# Use the SAME Python interpreter that's running this notebook
# to install the package — avoids version mismatch between
# system Python and the notebook kernel
import subprocess, sys
subprocess.check_call([
    sys.executable,       # e.g. /usr/bin/python3.11
    "-m", "pip",
    "install",
    "--upgrade",          # ensure we get the latest version
    "--quiet",            # suppress verbose pip output
    "llm-connector"
])

# llm_connector.cli contains the one-time project scaffolding tool
from llm_connector.cli import init_project

# Creates this folder structure in your working directory:
#   llm-connector/
#     conf/        ← provider config files go here
#     logs/        ← session logs written here automatically
#     .env.template ← copy this to .env and add your API keys
#
# Safe to call multiple times — skips creation if folder already exists
init_project()

In [ ]:
# --- Standard library type hints ---
# Dict, List, Optional are used to annotate function signatures
# and dataclass fields throughout the tutorial
# e.g. List[Dict[str, str]] = a list of {"role": ..., "content": ...} message dicts
from typing import Dict, List, Optional

# --- Dataclass support ---
# @dataclass auto-generates __init__, __repr__, __eq__ from field definitions
# field() is used when a default value needs to be a mutable object (e.g. a list)
# without field(default_factory=list), Python would share ONE list across all instances
from dataclasses import dataclass, field

# --- regex ---
# Used later to extract JSON blocks from raw LLM response strings
# e.g. finding ```json ... ``` fences or raw { } objects in text
import re

# --- JSON serialization ---
# Used to parse LLM output into Python dicts (tool calls, answers)
# and to serialize tool schemas into the system prompt
import json

# --- Timing ---
# Used by the Tracer to record how long each ReAct step takes
import time

# --- LLM Connector ---
# chat_completion()    → sends a message list to the configured LLM provider
#                        returns (response, prompt_tokens, completion_tokens, total_tokens, latency)
# cleanup_resources()  → closes connections and flushes logs at the end of the session
from llm_connector import chat_completion, cleanup_resources

## Testing the Setup

Select a provider and model, configure inference parameters, and run a single test call to confirm the API key is valid and the provider is reachable before building the agent.

In [ ]:
# --- Provider & Model Selection ---
# Only ONE provider/model pair should be active at a time
# Comment out all others
#
# IMPORTANT: a .env file must exist in llm-connector/ with the correct API key
# for whichever provider is active, e.g.:
#   GROQ_API_KEY=your_key_here
#   OPENROUTER_API_KEY=your_key_here
#   OPENAI_API_KEY=your_key_here
#   ANTHROPIC_API_KEY=your_key_here
#   GOOGLE_API_KEY=your_key_here
#
# Copy llm-connector/.env.template to llm-connector/.env and fill in your keys
# Missing or wrong key = authentication error at runtime

#PROVIDER = "groq"
#MODEL = "llama-3.3-70b-versatile"

PROVIDER = "openrouter"       # currently active provider
MODEL = "meta-llama/llama-3.2-3b-instruct"     # currently active model

#PROVIDER = "openai"
#MODEL = "gpt-5.4"

#PROVIDER = "anthropic"
#MODEL = "claude-sonnet-4-6"

#PROVIDER = "google"
#MODEL = "gemini-3.1-pro-preview"

# --- Inference Parameters ---
TEMPERATURE = 0.2   # low value = more deterministic, consistent outputs
                    # range 0.0-1.0; agents benefit from low values to reduce
                    # hallucinated tool calls
TOP_P = 0.7         # nucleus sampling — only consider tokens comprising
                    # the top 70% of probability mass; works alongside temperature
MAX_TOKENS = 1024   # hard cap on response length per turn
                    # keeps costs predictable and prevents runaway generation

# --- Sanity Check ---
# A minimal single-turn call to verify the provider is reachable
# and the API key is valid before running the full agent loop
messages = [{"role": "user", "content": "Hello! What is happening? What model are you?"}]

# chat_completion returns a 5-tuple:
#   response = the model's reply as a string
#   p        = prompt token count
#   c        = completion token count
#   t        = total token count
#   latency  = round-trip time in seconds
response, p, c, t, latency = chat_completion(messages, provider=(PROVIDER, MODEL))

print(response)

## Define an LLM Agent
Define AgentState to hold conversation history and BaseLLMAgent as a reusable base class. The agent appends messages, sends the full history to the LLM on every call, and stores the reply — it has no built-in memory beyond the message list.

In [ ]:
@dataclass
class AgentState:
    """Represents the current state of the agent."""
    # The full conversation history sent to the LLM on every call
    # Each entry is a dict with "role" and "content" keys
    # Roles: "system", "user", "assistant"
    messages: List[Dict[str, str]]

    # Stored separately so the system prompt can be inspected
    # or reset without scanning the messages list
    system_prompt: str


class BaseLLMAgent:
    """
    Base LLM agent — reusable for any prompt-driven loop.
    Named BaseLLMAgent to signal it is a general-purpose foundation,
    not tied to ReAct specifically. A ReAct agent is built ON TOP of
    this by controlling what messages get added and in what order.
    """

    def __init__(self, system_prompt: str):
        self.state = AgentState(
            # System prompt is the FIRST message in the list
            # It sets the agent's persona, rules, and available tools
            # Everything else appends after it
            messages=[{"role": "system", "content": system_prompt}],
            system_prompt=system_prompt,
        )

    def add_message(self, role: str, content: str) -> None:
        # Mutates the message list in place
        # Called with role="user" for questions and observations
        # Called with role="assistant" for model responses
        self.state.messages.append({"role": role, "content": content})

    def execute(self) -> str:
        # Sends the ENTIRE message history to the LLM on every call
        # The model has no memory — full context must be re-sent each time
        response, p, c, t, latency = chat_completion(
            self.state.messages,
            provider=(PROVIDER, MODEL),
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS
        )

        # Some providers return None on network failure or rate limiting
        # Guard here prevents NoneType errors downstream in the ReAct loop
        if response is None:
            response = "[No response from model]"

        return response

    def __call__(self, message: str) -> str:
        # __call__ makes the agent usable as a function: agent("your question")
        # Three steps always happen in this order:
        #   1. append the incoming message as "user"
        #   2. send full history to LLM via execute()
        #   3. append the model's reply as "assistant"
        # This keeps the conversation history symmetric and complete
        self.add_message("user", message)
        result = self.execute()
        self.add_message("assistant", result)
        return result

## Debug Utilities
Define TraceStep to capture one turn of the loop and Tracer to collect and pretty-print a summary table of all turns — showing thought, tool called, input, observation, and elapsed time for each step.

In [ ]:
@dataclass
class TraceStep:
    """One step of the ReAct loop, captured for analysis."""

    turn: int               # which iteration of the loop this belongs to
    thought: str            # the LLM's reasoning text extracted from that turn

    # All fields below are Optional — not every step calls a tool
    # e.g. the final "answer" step has no tool, input, or observation
    tool: Optional[str] = None          # name of the tool called, if any
    tool_input: Optional[str] = None    # argument passed to the tool
    observation: Optional[str] = None  # result returned by the tool
    timestamp: float = 0.0             # seconds elapsed since Tracer was created


class Tracer:
    """
    Passive observer — records what the agent does each turn.
    Does not affect agent behaviour, only captures it for inspection.
    Think of it as a flight recorder for the ReAct loop.
    """

    def __init__(self):
        self.steps: List[TraceStep] = []
        self.start_time: float = time.time()  # reference point for all timestamps

    def record(self, turn: int, thought: str, tool: Optional[str] = None,
               tool_input: Optional[str] = None, observation: Optional[str] = None) -> None:

        self.steps.append(TraceStep(
            turn=turn,
            thought=thought,
            tool=tool,
            tool_input=tool_input,
            observation=observation,
            # timestamp = elapsed time since this Tracer was created
            # not wall-clock time — relative to the start of THIS query
            timestamp=time.time() - self.start_time,
        ))

    @staticmethod
    def _truncate(text: Optional[str], width: int) -> str:
        # None means the field was not used this turn — display a dash
        if text is None:
            return "—"
        # Collapse newlines so multi-line LLM output fits in one table row
        text = text.replace("\n", " ").strip()
        # Trim to column width and add ellipsis to signal truncation
        return text if len(text) <= width else text[: width - 1] + "…"

    def summary(self) -> None:
        if not self.steps:
            print("(no trace steps recorded)")
            return

        # W defines the printable content width of each column
        # separators ( "| " ) are added on top of these widths
        W = {"turn": 6, "thought": 30, "tool": 18, "input": 20, "obs": 24, "time": 7}

        def row_str(turn, thought, tool, inp, obs, tm):
            # :< = left-align, {W['turn']} = pad to column width
            # every cell is padded to the same width so columns stay aligned
            return (f"| {turn:<{W['turn']}}| {thought:<{W['thought']}}| {tool:<{W['tool']}}"
                    f"| {inp:<{W['input']}}| {obs:<{W['obs']}}| {tm:<{W['time']}}|")

        # +2 per column accounts for "| " prefix on each cell
        # +1 accounts for the trailing "|" at the end of every row
        total_w = sum(W.values()) + len(W) * 2 + 1

        # Two divider styles:
        #   divider     = single solid line spanning full table width (header/footer)
        #   col_divider = per-column segmented line (between header row and data rows)
        divider = "+" + "-" * (total_w - 2) + "+"
        col_divider = (f"+{'-' * (W['turn'] + 1)}+{'-' * (W['thought'] + 1)}"
                       f"+{'-' * (W['tool'] + 1)}+{'-' * (W['input'] + 1)}"
                       f"+{'-' * (W['obs'] + 1)}+{'-' * (W['time'] + 1)}+")

        # --- Print table header ---
        print(f"\n{divider}")
        print(f"|{'TRACE SUMMARY':^{total_w - 2}}|")  # ^= centre-align title
        print(col_divider)
        print(row_str("Turn", "Thought", "Tool", "Input", "Observation", "Time"))
        print(col_divider)

        # --- Print one row per recorded TraceStep ---
        for step in self.steps:
            print(row_str(
                str(step.turn),
                self._truncate(step.thought, W["thought"] - 1),   # -1 leaves room for "…"
                self._truncate(step.tool, W["tool"] - 1),
                self._truncate(step.tool_input, W["input"] - 1),
                self._truncate(step.observation, W["obs"] - 1),
                f"{step.timestamp:.2f}s",
            ))

        print(col_divider)
        # last step's timestamp = total elapsed time for the full query
        print(f"Total steps: {len(self.steps)} | Total time: {self.steps[-1].timestamp:.2f}s\n")

## Tools
Define the two tool functions the agent can call — model_memory for looking up conversion rates and apply_conversion for performing the calculation — and declare their JSON schemas so the LLM knows what tools exist and what arguments to produce.

In [ ]:
# TOOL_SCHEMAS is a list of tool definitions written for the LLM, not for Python
# Python calls the actual functions directly — this list is injected into the
# system prompt so the LLM knows what tools exist and how to call them
#
# Each schema has three fields:
#   name        — must exactly match the key in KNOWN_ACTIONS dict
#   description — plain English; this is what the LLM reads to decide WHICH tool to use
#   parameters  — tells the LLM what arguments to produce and in what format

TOOL_SCHEMAS = [
    {
        "name": "model_memory",

        # LLM reads this to decide: "I need a conversion rate → use model_memory"
        "description": "Get the conversion rate for a unit pair (e.g., meters to feet, celsius to fahrenheit)",

        "parameters": {
            "unit": {
                "type": "string",
                # LLM reads this to know the exact string format expected
                # e.g. "meters to feet" not "m to ft" or "meter → foot"
                "description": "The conversion to look up, e.g. 'meters to feet'"
            }
        }
    },
    {
        "name": "apply_conversion",

        # LLM reads this to decide: "I have a rate and a value → use apply_conversion"
        # Description also encodes the TWO calling conventions the tool supports:
        #   regular:     "rate, value"           e.g. "3.28084, 5"
        #   temperature: "fraction, offset, value" e.g. "9/5, 32, 20"
        # Without this hint the LLM would not know which format to use
        "description": "Apply a conversion rate to a numeric value. For temperature use 'fraction, offset, value'. For others use 'rate, value'.",

        "parameters": {
            "params": {
                "type": "string",
                # Single string argument — the tool itself parses the commas internally
                # This keeps the schema simple at the cost of looser typing
                "description": "Comma-separated conversion string, e.g. '3.28084, 5' or '9/5, 32, 20'"
            }
        }
    }
]

In [ ]:
def model_memory(unit: str) -> str:
    # This is a SIMULATED memory tool — in a real agent this would be
    # a database lookup, an API call, or a vector store retrieval
    # Here it is a hardcoded dict to keep the tutorial self-contained

    rates = {
        "meters to feet":       "3.28084",
        "kilometers to miles":  "0.621371",
        "kilograms to pounds":  "2.20462",

        # Temperature is a special case — conversion requires TWO parameters:
        # a multiplier (9/5) AND an offset (+32), not just a single rate
        # Encoded as a single string "9/5,32" so apply_conversion can detect
        # the 3-part format when it splits on commas
        "celsius to fahrenheit": "9/5,32",
    }

    # .lower() makes the lookup case-insensitive
    # e.g. "Meters to Feet" and "meters to feet" both resolve correctly
    # If the unit is not in the dict, return a descriptive error string
    # — the agent will receive this as an Observation and can react to it
    return rates.get(unit.lower(), f"No conversion rate found for {unit}")


def apply_conversion(params: str) -> str:
    # params arrives as a raw string from the LLM, e.g. "3.28084, 100"
    # We split on commas to determine which of two formats was used:
    #   2 parts → regular conversion:     "rate, value"
    #   3 parts → temperature conversion: "fraction, offset, value"

    try:
        parts = params.split(",")

        if len(parts) == 3:
            # --- Temperature path ---
            # parts = ["9/5", "32", "20"]
            multiplier_str, offset, value_str = parts

            num = float(value_str.strip())    # the value to convert, e.g. 20.0

            # The multiplier is stored as a fraction string "9/5"
            # We manually split and divide instead of using eval()
            # NEVER use eval() on strings that came from an LLM or tool output —
            # eval() executes arbitrary Python code, creating a code injection risk
            frac_parts = multiplier_str.strip().split("/")
            multiplier = float(frac_parts[0]) / float(frac_parts[1])  # 9/5 = 1.8

            offset = float(offset.strip())    # 32.0
            result = num * multiplier + offset # standard °C → °F formula
            return f"{num}°C = {result:.2f}°F"

        elif len(parts) == 2:
            # --- Regular conversion path ---
            # parts = ["3.28084", "100"]
            rate_str, value_str = parts

            num = float(value_str.strip())
            conversion_rate = float(rate_str)
            result = num * conversion_rate

            # Reverse-lookup the unit names from the rate string
            # so the output message is human-readable rather than just a number
            # Falls back to generic "units" if the rate is not in the map
            unit_map = {
                "3.28084":  ("meters",     "feet"),
                "0.621371": ("kilometers", "miles"),
                "2.20462":  ("kilograms",  "pounds"),
            }
            source_unit, target_unit = unit_map.get(rate_str.strip(), ("units", "units"))
            return f"{num} {source_unit} = {result:.2f} {target_unit}"

        else:
            # LLM produced the wrong number of comma-separated arguments
            # Return an error string — the self-healing loop in query() will
            # feed this back to the LLM along with the correct schema
            return "Error: Invalid number of parameters"

    except ValueError:
        # float() failed — LLM passed a non-numeric string where a number was expected
        return "Error: Please provide valid numbers"

    except (IndexError, ZeroDivisionError):
        # IndexError  — fraction string did not contain "/"
        # ZeroDivisionError — denominator was 0 in the fraction
        return "Error: Invalid conversion rate format"

# Agent Initialization and Manual ReAct Loop
Walk through the ReAct loop turn-by-turn by hand: create an agent, send a question, manually call the tool the agent requested, inject the result as an Observation, and repeat until the agent produces a final answer. This section makes the loop mechanics visible before automation is introduced.

In [ ]:
def create_agent() -> BaseLLMAgent:
    """
    Create a conversion agent with the tutorial system prompt.
    The system prompt now includes JSON tool schemas and requires
    the LLM to output structured JSON for actions and answers.
    """
    # Format the tool schemas into a readable string for the prompt
    tools_json = json.dumps(TOOL_SCHEMAS, indent=2)

    system_prompt = f"""
    You run in a loop of Thought, Action, PAUSE, Observation.
    At the end of the loop you output an Answer.
    Use Thought to describe your thoughts about the question you have been asked.
    Use Action to run one of the actions available to you — then return PAUSE.
    Observation will be the result of running those actions.

    IMPORTANT — Output Format Rules:
    - When you want to call a tool, output your Thought as plain text, then
      output a JSON block on its own line with this exact structure:
      ```json
      {{"type": "action", "tool": "<tool_name>", "input": {{<arguments>}}}}
      ```
      Then write PAUSE on the next line.
    - When you have the final answer, output:
      ```json
      {{"type": "answer", "content": "<your final answer>"}}
      ```
    - Do NOT output any other JSON structure.
    - Always output exactly one JSON block per response.

    Available tools:
    {tools_json}

    Example session #1:
    Question: Convert 5 meters to feet?

    Thought: This is a length conversion from meters to feet. First, I need the conversion rate.
    ```json
    {{"type": "action", "tool": "model_memory", "input": {{"unit": "meters to feet"}}}}
    ```
    PAUSE

    You will be called again with this:

    Observation: 3.28084

    You then output:

    Thought: Now I can apply this conversion rate to 5 meters.
    ```json
    {{"type": "action", "tool": "apply_conversion", "input": {{"params": "3.28084, 5"}}}}
    ```
    PAUSE

    You will be called again with this:

    Observation: 5 meters = 16.40 feet

    You then output:
    ```json
    {{"type": "answer", "content": "5 meters is equal to 16.40 feet"}}
    ```

    Example session #2:
    Question: Convert 20°C to Fahrenheit?

    Thought: This is a temperature conversion. I need the conversion formula first.
    ```json
    {{"type": "action", "tool": "model_memory", "input": {{"unit": "celsius to fahrenheit"}}}}
    ```
    PAUSE

    Observation: 9/5,32

    Thought: Now I can apply this formula to 20°C.
    ```json
    {{"type": "action", "tool": "apply_conversion", "input": {{"params": "9/5,32, 20"}}}}
    ```
    PAUSE

    Observation: 20°C = 68.00°F

    ```json
    {{"type": "answer", "content": "20°C is equal to 68.00°F"}}
    ```
    """.strip()

    return BaseLLMAgent(system_prompt)

In [ ]:
# create_agent() builds a fresh BaseLLMAgent with the full system prompt
# including tool schemas and few-shot examples
# Every call to create_agent() starts a clean conversation history
agent = create_agent()

# Calling agent() triggers BaseLLMAgent.__call__():
#   1. appends "Convert 100 meters to feet?" as a "user" message
#   2. sends the full message list to the LLM via execute()
#   3. appends the LLM reply as an "assistant" message
#   4. returns the raw LLM response string
result = agent("Convert 100 meters to feet?")

# At this point the agent has taken ONE turn only
# The LLM should have responded with a Thought + action JSON + PAUSE
# It has NOT called any tools yet — that happens in the query() loop later
# This cell is a manual step-by-step walkthrough of the loop
# to show what happens inside each individual turn before
# the full automated loop is introduced
print(result)

# Expected output shape:
# Thought: This is a length conversion. I need the conversion rate.
# {"type": "action", "tool": "model_memory", "input": {"unit": "meters to feet"}}
# PAUSE

In [ ]:
# This cell manually executes what the agent REQUESTED in the previous turn
# The agent output an action JSON pointing to model_memory with "meters to feet"
# Here we are playing the role of the query() dispatcher — calling the tool ourselves

result = model_memory("meters to feet")

# result is a plain string: "3.28084"
# In the automated loop this string would be wrapped as:
# "Observation: 3.28084" and fed back to the agent as a "user" message
# Here we print it raw to inspect what the tool actually returns
print(result)

In [ ]:
# Wrap the tool output in the "Observation: " prefix
# This is the exact format the system prompt taught the LLM to expect
# Without the prefix the LLM may not recognise this as a tool result
# and could treat it as a new user question instead
next_prompt = "Observation: {}".format(result)
# next_prompt is now: "Observation: 3.28084"

# Feed the observation back into the agent as the next user message
# BaseLLMAgent.__call__() will:
#   1. append "Observation: 3.28084" as a "user" message
#   2. send the full history (system + turn1 + observation) to the LLM
#   3. append the LLM reply as "assistant"
# The LLM now has the rate it needed and should respond with
# a second action — calling apply_conversion with "3.28084, 100"
result = agent(next_prompt)

print(result)

# Expected output shape:
# Thought: Now I can apply this conversion rate to 100 meters.
# {"type": "action", "tool": "apply_conversion", "input": {"params": "3.28084, 100"}}
# PAUSE

In [ ]:
# Again playing the role of the dispatcher manually —
# the agent requested apply_conversion with "3.28084, 100"
# so we call it directly to see what the tool returns

result = apply_conversion("3.28084, 100")

# apply_conversion splits "3.28084, 100" on commas → 2 parts → regular conversion path
# multiplies 100 * 3.28084 and looks up "3.28084" in unit_map
# returns a human-readable string, not a number

# No print() here — in a notebook the last expression on a cell
# is automatically displayed as the cell output
# result = "100 meters = 328.08 feet"
result

In [ ]:
# Wrap the tool result the same way as before
# The LLM expects every tool result to arrive prefixed with "Observation: "
next_prompt = "Observation: {}".format(result)
# next_prompt is now: "Observation: 100 meters = 328.08 feet"

# Feed the final observation back into the agent
# The LLM now has everything it needs:
#   Turn 1 — asked for conversion rate → got 3.28084
#   Turn 2 — asked to apply rate to 100 → got 100 meters = 328.08 feet
# It should now exit the loop by outputting type=answer

agent(next_prompt)

# Expected output shape:
# {"type": "answer", "content": "100 meters is equal to 328.08 feet"}

# Note: no print() or variable assignment here — the notebook displays
# the return value of agent() directly as the cell output
# This also means the answer is NOT stored — next cell inspects full history instead

In [ ]:
# Print the complete conversation history accumulated across all manual turns
# This is the exact list that gets sent to the LLM on every execute() call
# Reading it shows how the ReAct loop looks from the LLM's perspective

print(agent.state.messages)

# Expected structure:
# [
#   {"role": "system",    "content": "You run in a loop of Thought, Action, PAUSE..."},
#   {"role": "user",      "content": "Convert 100 meters to feet?"},
#   {"role": "assistant", "content": "Thought: ... action JSON ... PAUSE"},
#   {"role": "user",      "content": "Observation: 3.28084"},
#   {"role": "assistant", "content": "Thought: ... apply_conversion JSON ... PAUSE"},
#   {"role": "user",      "content": "Observation: 100 meters = 328.08 feet"},
#   {"role": "assistant", "content": '{"type": "answer", "content": "..."}'}
# ]

# Key observations:
# — the system prompt is always index 0, never changes
# — user and assistant messages alternate strictly
# — observations arrive as "user" role — the LLM cannot distinguish
#   them from real human input, which is exactly why sanitization matters
# — the full history grows with every turn — this is the agent's only memory

# Running Everythin in the ReAct Loop
Automate the full loop using react_loop() and wrap it with query() as the public entry point. This section also defines KNOWN_ACTIONS as the tool dispatcher registry and extract_json_block() for parsing LLM output.

## Defining Available Actions and Extraction Strategy

Register tool names to Python callables in KNOWN_ACTIONS and implement extract_json_block() which tries two strategies — fenced code block first, raw brace fallback second — to extract structured JSON from the LLM's raw response string.

In [ ]:
# KNOWN_ACTIONS is the dispatcher registry —
# it maps the tool name strings the LLM produces in its JSON output
# to the actual callable Python functions
# When the LLM outputs {"type": "action", "tool": "model_memory", ...}
# query() does: KNOWN_ACTIONS["model_memory"](arg)
# Adding a new tool = add the function + add an entry here + add a schema to TOOL_SCHEMAS
KNOWN_ACTIONS = {
    "model_memory":     model_memory,
    "apply_conversion": apply_conversion,
}


def extract_json_block(text: str) -> dict | None:
    # The LLM returns a raw string — we need to find the JSON inside it
    # Two strategies are tried in order of reliability

    # --- Strategy 1: fenced code block ---
    # The system prompt instructs the LLM to wrap JSON in ```json ... ```
    # This is the expected happy-path format
    # re.DOTALL makes . match newlines — JSON blocks often span multiple lines
    # group(1) captures only the { } content, not the surrounding backticks
    fence_match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence_match:
        try:
            return json.loads(fence_match.group(1))
        except json.JSONDecodeError:
            # Regex matched but content was not valid JSON
            # e.g. LLM hallucinated a malformed structure inside the fence
            # fall through to strategy 2 instead of crashing
            pass

    # --- Strategy 2: raw JSON object anywhere in text ---
    # Fallback for when the LLM drops the fences but still outputs valid JSON
    # (\{.*\}) is greedy — matches the LARGEST { } span in the text
    # risk: could accidentally match a JSON-like substring that is not the action
    # this is why strategy 1 is preferred — fences give an unambiguous boundary
    brace_match = re.search(r"(\{.*\})", text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(1))
        except json.JSONDecodeError:
            # Found braces but content still not valid JSON
            # nothing more we can do — return None and let query() handle it
            pass

    # Neither strategy found valid JSON
    # query() treats None as a plain-text final answer (fallback path)
    return None

## Safety Preprocessing
Implement sanitize_observation() to strip agent command patterns from tool output before it is injected back into the conversation, preventing prompt injection at the observation boundary.

In [ ]:
def sanitize_observation(text: str) -> str:
    # Tool output arrives as a plain string and gets injected into the
    # conversation as a "user" role message — the LLM cannot distinguish
    # it from a real human message or its own prior instructions
    # This creates an injection boundary: untrusted content entering trusted context

    # --- Defense 1: remove fenced JSON blocks ---
    # Targets the exact format the system prompt teaches the LLM to produce:
    #   ```json { "type": ... } ```
    # If a tool returns this pattern the LLM would parse it as its own action
    # re.DOTALL covers multi-line JSON blocks
    # Entire match replaced with [REDACTED] — content gone, boundary marked
    text = re.sub(r'```json\s*\{.*?\}\s*```', '[REDACTED]', text, flags=re.DOTALL)

    # --- Defense 2: break the raw JSON trigger pattern ---
    # Targets {"type": anywhere in the text without fences
    # This is weaker than Defense 1 — it does NOT remove the content,
    # it only breaks the opening pattern so json.loads() will fail on it
    # {"type":   →   [REDACTED: {"type":
    # The rest of the JSON string remains visible in the conversation
    # which preserves evidence of the injection attempt for debugging
    # A determined attacker can bypass this with a space: { "type":
    # This is illustrative — not a production-grade defense
    text = re.sub(r'\{"type"\s*:', '[REDACTED: {"type":', text)

    # Returns the sanitized string — always called in query() before
    # the observation is wrapped in "Observation: " and fed back to the agent
    return text


# IMPORTANT: ReAct Loop

The core react_loop() function — runs the full automated Thought→Action→Observation cycle, handles tool dispatch, self-healing on failures, observation sanitization, max-turn guard, and trace recording.

In [ ]:
def react_loop(question: str, max_turns: int = 5) -> List[Dict[str, str]]:
    """
    Process a question through multiple turns of the ReAct loop.

    Features:
      - JSON-based tool calling (parsed from LLM output)
      - Structured tracing via Tracer
      - Observation sanitization against prompt injection
      - Self-healing: on tool failure, feeds error + tool schema back
        so the LLM can self-correct its arguments (1 retry per turn)

    Args:
        question: The user's question
        max_turns: Maximum number of turns to process (prevents infinite loops)

    Returns:
        The complete conversation history
    """

    # Fresh agent and tracer per question — no state bleeds between calls
    agent = create_agent()
    tracer = Tracer()

    # First prompt is the raw question — subsequent prompts are Observations
    next_prompt = question

    for turn in range(max_turns):

        # Send current prompt to the LLM and get raw response string
        # On turn 0: next_prompt = the original question
        # On turn N: next_prompt = "Observation: <tool result>"
        result = agent(next_prompt)
        print(result)

        # Pull out the Thought line for the tracer
        # The tracer only stores a snippet — not the full LLM response
        thought_match = re.search(r"Thought:\s*(.+)", result)
        thought_text = thought_match.group(1).strip() if thought_match else ""

        # Attempt to extract a JSON block from the LLM response
        # Returns a dict if found, None if the response was plain text
        parsed = extract_json_block(result)

        if parsed is None:
            # LLM responded with plain text and no JSON
            # Treat the full response as the final answer — graceful fallback
            # This can happen when the LLM ignores the output format instructions
            tracer.record(turn + 1, thought_text or result)
            print(f"\n>>> Final Answer (no JSON detected): {result}")
            break

        msg_type = parsed.get("type")

        if msg_type == "answer":
            # LLM signalled it is done — extract the answer content and exit
            answer = parsed.get("content", result)
            tracer.record(turn + 1, thought_text, observation=f"[ANSWER] {answer}")
            print(f"\n>>> Final Answer: {answer}")
            break

        elif msg_type == "action":
            tool_name  = parsed.get("tool")
            tool_input = parsed.get("input", {})

            print(f"INFO: Detected action: {tool_name} with input: {tool_input}")

            if tool_name not in KNOWN_ACTIONS:
                # LLM hallucinated a tool name that does not exist in the registry
                # Feed the error back so the LLM can self-correct on the next turn
                print(f"WARNING: Unknown tool '{tool_name}' — feeding error back.")
                tracer.record(turn + 1, thought_text, tool=tool_name,
                              observation=f"ERROR: unknown tool")
                next_prompt = (f"Observation: Error — unknown tool '{tool_name}'. "
                               f"Available tools are: {', '.join(KNOWN_ACTIONS.keys())}")
                continue

            # tool_input is a dict e.g. {"unit": "meters to feet"}
            # Our tools each take a single positional string argument
            # so we extract the first value regardless of the key name
            arg_value = list(tool_input.values())[0] if tool_input else ""

            print(f"\n ---> Executing {tool_name}({arg_value})")
            try:
                observation = KNOWN_ACTIONS[tool_name](arg_value)

            except Exception as e:
                # --- Self-Healing ---
                # Instead of crashing, feed the exception message AND the
                # tool's full schema back to the LLM as the next Observation
                # The LLM can read the schema hint and retry with correct arguments
                # One retry per turn — if it fails again it will fail on the next turn too
                tool_schema = next(
                    (t for t in TOOL_SCHEMAS if t["name"] == tool_name), None
                )
                observation = (
                    f"Error executing {tool_name}: {e}\n"
                    f"Hint — expected schema: {json.dumps(tool_schema, indent=2)}"
                )
                print(f"SELF-HEAL: Feeding error + schema back to agent")

            # --- Observation Sanitization ---
            # Always sanitize before injecting tool output back into the conversation
            # Converts observation to str first in case the tool returned a non-string
            observation = sanitize_observation(str(observation))

            print(f"Observation: {observation}")
            tracer.record(turn + 1, thought_text, tool=tool_name,
                          tool_input=arg_value, observation=observation)

            # Wrap in "Observation: " prefix — this is what the system prompt
            # taught the LLM to expect as the result of a tool call
            next_prompt = f"Observation: {observation}"

        else:
            # LLM produced a JSON block with a type other than "action" or "answer"
            # Feed the error back and let the LLM self-correct the format
            tracer.record(turn + 1, thought_text,
                          observation=f"ERROR: unrecognized type '{msg_type}'")
            print(f"WARNING: Unrecognized JSON type '{msg_type}'")
            next_prompt = (f"Observation: Error — unrecognized response type '{msg_type}'. "
                           f"Use 'action' or 'answer'.")

    else:
        # The for loop completed all max_turns without hitting a break
        # meaning no "answer" or fallback exit was reached
        # "else" on a for loop runs only when the loop was NOT broken out of
        print(f"\nWARNING: Reached maximum turns ({max_turns}) without a final answer.")

    # Print the full trace table — turn, thought, tool, input, observation, time
    tracer.summary()

    # Return full message history for inspection or downstream use
    return agent.state.messages

In [ ]:
## Loop Wrapper to Stress the Structure

In [ ]:
def query(initial_prompt: str, max_turns: int = 5) -> List[Dict[str, str]]:
    """
    Entry point for running a question through the ReAct loop.

    Args:
        initial_prompt: The user's question or task
        max_turns: Maximum number of turns passed to react_loop

    Returns:
        The complete conversation history
    """
    # Thin wrapper — formats the prompt and delegates to react_loop
    # Separates the concern of prompt preparation from loop execution
    # Allows future pre-processing here (e.g. input validation, logging,
    # prompt templating) without touching react_loop internals
    return react_loop(initial_prompt, max_turns=max_turns)

# Examples
Five worked examples demonstrating the agent in action: length, temperature, weight, and distance conversions as happy-path cases, and an unsupported unit conversion to demonstrate graceful failure and self-healing behaviour.

In [ ]:
# --- Example 1: Length Conversion ---
# Demonstrates the happy-path two-tool sequence:
#   Turn 1: model_memory("meters to feet")      → "3.28084"
#   Turn 2: apply_conversion("3.28084, 10")     → "10 meters = 32.81 feet"
#   Turn 3: type=answer                         → loop exits

print("\nExample 1: Length Conversion")
print("=" * 50)

question = "Convert 10 meters to feet"

# query() wraps react_loop() — starts a fresh agent and tracer internally
# result holds the full conversation history on return
result = query(question)

print("\n" + "=" * 50 + "\n")

# Expected trace summary:
# Turn 1 | model_memory     | meters to feet  | 3.28084
# Turn 2 | apply_conversion | 3.28084, 10     | 10 meters = 32.81 feet
# Turn 3 | —                | —               | [ANSWER] 10 meters is equal to 32.81 feet

In [ ]:
# Example 2: Temperature conversion
print("\nExample 2: Temperature Conversion")
print("=" * 50)
question = "Convert 20°C to Fahrenheit"
result = query(question)
print("\n" + "=" * 50 + "\n")

In [ ]:
# Example 3: Weight conversion
print("\nExample 3: Weight Conversion")
print("=" * 50)
question = "Convert 10 kilograms to pounds"
result = query(question)
print("\n" + "=" * 50 + "\n")

In [ ]:

# Example 4: Distance conversion
print("\nExample 4: Distance Conversion")
print("=" * 50)
question = "Convert 3 kilometers to miles"
result = query(question)
print("\n" + "=" * 50 + "\n")

In [ ]:
# Example 5: Unsupported conversion (demonstrates failure / self-healing)
print("\nExample 5: Unsupported Conversion (Failure Case)")
print("=" * 50)
question = "Convert 5 gallons to liters"
result = query(question)
print("\n" + "=" * 50 + "\n")

# Cleaning Up & Checking Logs
Call cleanup_resources() to close provider connections and flush session logs, then read and print the session summary block from the most recent log file.

In [ ]:
# Signals the llm_connector to gracefully shut down
# Two things happen internally:
#   1. Open network connections to the LLM provider are closed
#   2. Any buffered log entries are flushed to llm-connector/logs/
# Always call this at the end of a session — skipping it can result
# in incomplete log files or hanging connections
cleanup_resources()

## Logs
Parse the latest log file written by llm-connector, locate the Session Summary block, strip the timestamp prefix from each line, and print the clean summary to the notebook output.

In [ ]:
import os, glob

# Build the path to the logs directory created by init_project()
# os.path.join is OS-safe — works on Windows and Unix without hardcoding separators
# os.getcwd() anchors the path to wherever the notebook is running from
log_dir = os.path.join(os.getcwd(), "llm-connector", "logs")

# glob finds all .log files in the directory and sorted() orders them
# chronologically — log filenames contain timestamps so alphabetical
# order equals time order, making [-1] reliably the most recent session
log_files = sorted(glob.glob(os.path.join(log_dir, "*.log")))

if log_files:
    latest_log = log_files[-1]
    print(f"\n📄 Session log: {latest_log}\n")

    with open(latest_log, "r") as f:
        lines = f.readlines()

    # We only want the Session Summary block at the end of the log
    # not the full verbose per-call entries above it
    # in_summary acts as a gate — False until the marker line is found
    in_summary = False

    for line in lines:

        # Detect the start of the summary block by its marker string
        if "Session Summary" in line:
            in_summary = True

        if in_summary:
            # Each log line is prefixed with timestamp and logger name:
            # "2026-03-20 14:22:01 - llm_connector - INFO - <content>"
            # split(" - ", 3) splits on the first 3 occurrences of " - "
            # parts[-1] discards the prefix and keeps only the content
            # If the line has fewer than 3 separators, fall back to the raw line
            parts = line.split(" - ", 3)
            content = parts[-1] if len(parts) >= 3 else line
            print(content, end="")  # end="" preserves original line endings

        # Detect the end of the summary block — a "---" divider line
        # that is NOT the opening "Session Summary" line itself
        if in_summary and "---" in line and "Session Summary" not in line:
            in_summary = False

else:
    # No logs found — likely cleanup_resources() was not called
    # or init_project() was never run in this working directory
    print("No log files found in", log_dir)

**Your ReAct agent is now operational with capabilities to:**
- Navigate complex problems using structured reasoning cycles
- Connect with and utilize external tools
- Preserve dialogue history and context
- Break down and solve multi-stage challenges